Create an ensemble voting classifier by manually implementing hard and soft voting mechanisms, dynamically weighting model contributions based on performance metrics without using predefined ensemble libraries.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

print("Dataset shape:", df.shape)
print("Target values:", np.unique(y))

Dataset shape: (79, 6)
Target values: [0 1]


In [2]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 47
Validation samples: 16
Testing samples: 16


In [3]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully")

Logistic Regression trained successfully
Decision Tree trained successfully
Random Forest trained successfully


In [4]:
validation_scores = {}

for name, model in models.items():
    pred = model.predict(X_val)
    score = accuracy_score(y_val, pred)
    validation_scores[name] = score

print("Validation accuracy:")
for name, score in validation_scores.items():
    print(name, ":", round(score, 4))

total_score = sum(validation_scores.values())

weights = {
    name: score / total_score
    for name, score in validation_scores.items()
}

print("\nDynamic model weights:")
for name, weight in weights.items():
    print(name, ":", round(weight, 4))

Validation accuracy:
Logistic Regression : 0.8125
Decision Tree : 0.6875
Random Forest : 0.8125

Dynamic model weights:
Logistic Regression : 0.3514
Decision Tree : 0.2973
Random Forest : 0.3514


In [5]:
hard_predictions = np.array([
    model.predict(X_test)
    for model in models.values()
])

hard_voting = []

for i in range(X_test.shape[0]):
    votes = hard_predictions[:, i]
    counts = np.bincount(votes, minlength=2)
    hard_voting.append(np.argmax(counts))

hard_voting = np.array(hard_voting)

print("Hard voting completed")
print("Sample predictions:", hard_voting[:10])

Hard voting completed
Sample predictions: [1 1 1 1 1 1 1 1 1 1]


In [6]:
probabilities = np.array([
    model.predict_proba(X_test)[:, 1]
    for model in models.values()
])

average_probabilities = np.mean(probabilities, axis=0)

soft_voting = (average_probabilities >= 0.5).astype(int)

print("Soft voting completed")
print("Sample predictions:", soft_voting[:10])

Soft voting completed
Sample predictions: [1 1 1 1 1 1 1 1 1 1]


In [7]:
model_weights = np.array(list(weights.values()))

weighted_probabilities = np.average(
    probabilities, axis=0, weights=model_weights
)

weighted_voting = (weighted_probabilities >= 0.5).astype(int)

print("Weighted voting completed")
print("Sample predictions:", weighted_voting[:10])

Weighted voting completed
Sample predictions: [1 1 1 1 1 1 1 1 1 1]


In [8]:
def evaluate_model(name, y_true, y_pred):
    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1 Score:", round(f1_score(y_true, y_pred, zero_division=0), 4))

for name, model in models.items():
    evaluate_model(name, y_test, model.predict(X_test))

evaluate_model("Hard Voting", y_test, hard_voting)
evaluate_model("Soft Voting", y_test, soft_voting)
evaluate_model("Weighted Soft Voting", y_test, weighted_voting)


 Logistic Regression
Accuracy: 0.75
Precision: 0.7857
Recall: 0.9167
F1 Score: 0.8462

 Decision Tree
Accuracy: 0.75
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571

 Random Forest
Accuracy: 0.75
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571

 Hard Voting
Accuracy: 0.75
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571

 Soft Voting
Accuracy: 0.75
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571

 Weighted Soft Voting
Accuracy: 0.75
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571


In [9]:
results = []

for name, model in models.items():
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred, zero_division=0)
    })

for name, pred in [
    ("Hard Voting", hard_voting),
    ("Soft Voting", soft_voting),
    ("Weighted Soft Voting", weighted_voting)
]:
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("Accuracy", ascending=False).to_string(index=False))

               Model  Accuracy  F1 Score
 Logistic Regression      0.75  0.846154
       Decision Tree      0.75  0.857143
       Random Forest      0.75  0.857143
         Hard Voting      0.75  0.857143
         Soft Voting      0.75  0.857143
Weighted Soft Voting      0.75  0.857143
